# 小时级模型特征工程：历史需求特征

目标：在原有日期、小时和天气特征基础上，加入**过去已经观测到的租赁量**，观察是否能改善小时级预测。

注意：这里的滞后特征只使用当前时刻之前的 `cnt`。在真实部署时，系统需要持续保存历史租赁量；因此本实验先用于验证模型改进潜力，不会立即替换网页模型。

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

DATA_PATH = Path('../data/hour.csv')
BASE_FEATURES = [
    'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
    'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
]

df = pd.read_csv(DATA_PATH, parse_dates=['dteday'])
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)
df.shape

## 1. 构造滞后特征

- `lag_1`：前一小时的实际租赁量；
- `lag_24`：前一天同一小时的实际租赁量；
- `lag_168`：前一周同一小时的实际租赁量；
- `rolling_mean_24`：过去 24 小时的平均租赁量。

所有特征都先 `shift(1)` 再滚动计算，确保不会把当前小时的目标值泄漏给模型。

In [ ]:
feature_df = df.copy()
feature_df['lag_1'] = feature_df['cnt'].shift(1)
feature_df['lag_24'] = feature_df['cnt'].shift(24)
feature_df['lag_168'] = feature_df['cnt'].shift(168)
feature_df['rolling_mean_24'] = feature_df['cnt'].shift(1).rolling(24).mean()

LAG_FEATURES = ['lag_1', 'lag_24', 'lag_168', 'rolling_mean_24']
feature_df = feature_df.dropna(subset=LAG_FEATURES).reset_index(drop=True)
print(f'构造特征后可用样本数：{len(feature_df)}')
feature_df[['dteday', 'hr', 'cnt'] + LAG_FEATURES].head()

## 2. 保持时间顺序进行对比

使用相同的随机森林配置，分别训练“基础特征模型”和“加入历史需求特征的模型”。前 80% 用于训练，后 20% 作为最终测试集。

In [ ]:
split_index = int(len(feature_df) * 0.8)
train = feature_df.iloc[:split_index]
test = feature_df.iloc[split_index:]

def evaluate(name, features):
    model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=1)
    model.fit(train[features], train['cnt'])
    prediction = model.predict(test[features])
    result = {
        '模型': name,
        'MAE': mean_absolute_error(test['cnt'], prediction),
        'RMSE': root_mean_squared_error(test['cnt'], prediction),
        'R²': r2_score(test['cnt'], prediction),
    }
    return model, prediction, result

base_model, base_prediction, base_result = evaluate('基础特征随机森林', BASE_FEATURES)
lag_model, lag_prediction, lag_result = evaluate('加入历史需求特征的随机森林', BASE_FEATURES + LAG_FEATURES)

pd.DataFrame([base_result, lag_result]).set_index('模型').round(3)

## 3. 解释新模型

若加入历史需求特征后指标提升，说明历史租赁量携带了当前天气和日期特征难以完全表达的短期变化信息。

In [ ]:
importance = (
    pd.Series(lag_model.feature_importances_, index=BASE_FEATURES + LAG_FEATURES)
    .sort_values(ascending=False)
    .to_frame('特征重要性')
)
importance

## 小结（运行后填写）

- 加入历史需求特征后，MAE / RMSE / R² 的变化：……
- 最重要的新特征：……
- 结论：若模型提升，说明历史需求信息对短期预测有价值；但部署时必须获得前一小时、前一天和前一周的真实租赁量。
- 下一步：可把历史租赁量作为网页的可选输入，或接入真实历史数据服务后再部署该模型。